In [1]:
import pandas as pd
import os
from pathlib import Path

os.chdir('..')
parent_dir = Path.cwd()

pd.options.display.float_format = '{:1.2f}'.format
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [2]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage = mortgage[mortgage['observation_date'] >= '2024-01-01'].reset_index().drop(columns=['index'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [3]:
mortgage.head()

,date,rate_30yr_fixed
0,2024-01-04,6.62
1,2024-01-11,6.66
2,2024-01-18,6.60
3,2024-01-25,6.69
4,2024-02-01,6.63


In [4]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
 mortgage.groupby('year_month')['rate_30yr_fixed']
 .mean()
 .reset_index()
)
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,2024-01,6.64
1,2024-02,6.78
2,2024-03,6.82
3,2024-04,6.99
4,2024-05,7.06


In [5]:
sold = pd.read_csv(str(parent_dir)+'/week 1/sold_merged_residential.csv', encoding='latin-1')
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
sold[['CloseDate', 'year_month']].head()

C:\Users\Jamesb\AppData\Local\Temp\ipykernel_23544\3543926219.py:1: DtypeWarning: Columns (0,1,9,64,78,79,80,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(str(parent_dir)+'/week 1/sold_merged_residential.csv', encoding='latin-1')


,CloseDate,year_month
0,2024-01-26,2024-01
1,2024-01-05,2024-01
2,2024-01-05,2024-01
3,2024-01-30,2024-01
4,2024-01-29,2024-01


In [6]:
listings = pd.read_csv(str(parent_dir)+'/week 1/listing_merged_residential.csv', encoding='latin-1')
listings['year_month'] = pd.to_datetime(
    listings['ListingContractDate']
).dt.to_period('M')
listings[[ 'ListingContractDate','year_month']].head()

C:\Users\Jamesb\AppData\Local\Temp\ipykernel_23544\1923958767.py:1: DtypeWarning: Columns (2,43) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(str(parent_dir)+'/week 1/listing_merged_residential.csv', encoding='latin-1')


,ListingContractDate,year_month
0,2024-01-01,2024-01
1,2024-01-24,2024-01
2,2024-01-12,2024-01
3,2024-01-20,2024-01
4,2024-01-12,2024-01


In [7]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [8]:
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [9]:
sold_with_rates[
    ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.00,6.64
1,2024-01-05,2024-01,815000.00,6.64
2,2024-01-05,2024-01,810000.00,6.64
3,2024-01-30,2024-01,858000.00,6.64
4,2024-01-29,2024-01,1890500.00,6.64


In [10]:
sold_with_rates[
    ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
].drop_duplicates('year_month').head(100)

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.00,6.64
11203,2024-02-07,2024-02,1249000.00,6.78
24266,2024-03-06,2024-03,1290000.00,6.82
40050,2024-04-02,2024-04,250000.00,6.99
57345,2024-05-31,2024-05,3300000.00,7.06
75819,2024-06-28,2024-06,4250000.00,6.92
92444,2024-07-31,2024-07,929000.00,6.85
110253,2024-08-09,2024-08,210000.00,6.50
127044,2024-09-12,2024-09,1090000.00,6.18
141599,2024-10-04,2024-10,208000.00,6.43


In [12]:
listings_with_rates[
    ['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']
].drop_duplicates('year_month').head(100)

,ListingContractDate,year_month,ListPrice,rate_30yr_fixed
0,2024-01-01,2024-01,1340000.00,6.64
17007,2024-02-18,2024-02,799999.00,6.78
34497,2024-03-27,2024-03,3550000.00,6.82
54998,2024-04-24,2024-04,549000.00,6.99
79023,2024-05-10,2024-05,929000.00,7.06
104470,2024-06-11,2024-06,699000.00,6.92
127780,2024-07-31,2024-07,949000.00,6.85
150799,2024-08-18,2024-08,300000.00,6.50
173014,2024-09-17,2024-09,570000.00,6.18
195271,2024-10-22,2024-10,788000.00,6.43


In [ ]:
# listings_with_rates.to_csv(str(parent_dir)+'/week 2-3/FRED_rates_listings.csv', index=False)
# sold_with_rates.to_csv(str(parent_dir)+'/week 2-3/FRED_rates_sold.csv', index=False)